# Préprocessing — DVF+ et Filosofi

**Projet :** Compagnon Immobilier · **Auteur :** Cécile NAUDET · **Relecteur :** _ · **Date :** 12/10/2026 (mis à jour le 15/10/2026)

Ce notebook prépare les données avant l'analyse. Il comporte trois parties :

1. **Échantillon DVF+** : filtrer les ventes d'un seul logement, puis tirer un échantillon d'apprentissage.
2. **Table cible** : calculer le prix médian au m² par commune et par année, sur toutes les ventes.
3. **Filosofi** : assembler les revenus des ménages par commune, de 2014 à 2021.

# Partie 1 — Échantillon DVF+

### Ce que l'audit a montré

L'audit a porté sur un échantillon de 100 000 mutations, tiré proportionnellement au poids de chaque département dans les 97 fichiers bruts (16,5 millions de mutations, 2014-2025, graine 42).

- **Une base hétérogène** : 92 % de ventes, 6,5 % de VEFA, le reste en échanges, adjudications et ventes de terrains à bâtir. Côté biens, « une maison » représente 34 % des mutations et « un appartement » 28 %. Le reste regroupe des terrains, des locaux d'activité et des mutations portant sur plusieurs biens.
- **Une valeur foncière par mutation, pas par bien.** Le prix ne se rapporte à un bien identifiable que si la mutation porte sur un seul logement.
- **Une qualité satisfaisante sur le résidentiel** (61 834 mutations) : aucun doublon, valeur foncière manquante dans 0,03 % des cas, surfaces bâties cohérentes.

L'audit a conduit à trois décisions :
- **Le périmètre** : ventes d'une seule maison ou d'un seul appartement, avec d'éventuelles dépendances, sur une seule commune, VEFA exclues.
- **La couverture** : 2014-2025, avec 2025 réservée au test ; France métropolitaine hors DOM (l'Alsace-Moselle est absente de DVF+).
- **Les variables** : 11 variables vides, constantes, redondantes ou inutiles pour le modèle ne sont pas conservées dans l'échantillon de travail.

### Pourquoi refaire un échantillon

- **L'échantillon d'audit a été tiré avant le filtrage.** Une fois réduit au résidentiel, il laisse trop peu de ventes par département et par année : la Lozère n'avait que 150 mutations au total, tous types confondus. C'est insuffisant pour étudier les prix par territoire et dans le temps.
- **La part du résidentiel varie fortement selon les départements.** Dans les départements ruraux, une grande partie des mutations porte sur des terrains. Tirer avant de filtrer surreprésenterait donc ces départements dans l'échantillon. Il faut filtrer d'abord, puis tirer. 
- **L'échantillon d'audit ne respecte pas le périmètre retenu** : il contient les VEFA, les DOM et les mutations portant sur plusieurs communes.

### Préparer les chemins et les liste de fichiers

In [19]:
# Imports
from pathlib import Path
import numpy as np
import pandas as pd

# Chemins : fichiers sources et dossier de sortie de la population filtrée
dossier = Path("/home/naude/formation/projet_compagnon/data/DVF+")
dossier_pop = dossier.parent / "dvf_plus_residentiel"

# Liste des fichiers départementaux, sans les DOM (hors périmètre)
DOM = {"dvf_plus_d971", "dvf_plus_d972", "dvf_plus_d973", "dvf_plus_d974"}
fichiers = [f for f in sorted(dossier.glob("dvf_plus_d*.csv")) if f.stem not in DOM]

# Contrôle : 93 fichiers attendus (97 - 4 DOM)
print("Nombre de fichiers retenus :", len(fichiers))
print("Premier :", fichiers[0].name, "| Dernier :", fichiers[-1].name)

Nombre de fichiers retenus : 93
Premier : dvf_plus_d01.csv | Dernier : dvf_plus_d95.csv


### Ecrire les règles de filtrage

In [20]:
def filtre_perimetre(df):
    """Garde les ventes d'un seul logement ancien, sur une seule commune."""
    # Vente classique uniquement (la VEFA a son propre libellé)
    nature_ok = df["libnatmut"] == "Vente"
    # Une seule commune, pour que l_codinsee contienne un code unique
    une_commune = df["nbcomm"] == 1
    # Une seule maison, sans appartement ni local d'activité (dépendances acceptées)
    maison_ok = (
        (df["libtypbien"] == "UNE MAISON")
        & (df["nblocmai"] == 1)
        & (df["nblocapt"] == 0)
        & (df["nblocact"] == 0)
    )
    # Un seul appartement, sans maison ni local d'activité (dépendances acceptées)
    appartement_ok = (
        (df["libtypbien"] == "UN APPARTEMENT")
        & (df["nblocapt"] == 1)
        & (df["nblocmai"] == 0)
        & (df["nblocact"] == 0)
    )
    return df.loc[nature_ok & une_commune & (maison_ok | appartement_ok)]

### Tester la fonction sur l'audit d'échantillon

In [21]:
# Test de la fonction sur l'échantillon d'audit (100 000 lignes, déjà connu)
audit = pd.read_csv(dossier.parent / "dvf_plus_echantillon_100k.csv", sep="|",
                    dtype={"coddep": str, "l_codinsee": str}, low_memory=False)

test = filtre_perimetre(audit)

# Contrôles attendus : moins de 61 834 lignes, uniquement « Vente »,
# uniquement les deux types de logement, nbcomm maximum = 1
print("Lignes retenues :", len(test))
print(test["libnatmut"].value_counts())
print(test["libtypbien"].value_counts())
print("nbcomm max :", test["nbcomm"].max())

Lignes retenues : 59155
libnatmut
Vente    59155
Name: count, dtype: int64
libtypbien
UNE MAISON        33800
UN APPARTEMENT    25355
Name: count, dtype: int64
nbcomm max : 1


### Traiter un fichier test (departement 48)

In [22]:
# Essai complet sur un seul département, le plus petit (Lozère)
fichier = dossier / "dvf_plus_d48.csv"
types_texte = {"coddep": str, "l_codinsee": str}  # codes géographiques lus en texte

# Lecture par blocs, filtrage de chaque bloc, puis assemblage
morceaux = []
n_brut = 0
for bloc in pd.read_csv(fichier, sep="|", chunksize=200_000,
                        dtype=types_texte, low_memory=False):
    n_brut += len(bloc)
    morceaux.append(filtre_perimetre(bloc))
pop = pd.concat(morceaux, ignore_index=True)

# Écriture en Parquet
dossier_pop.mkdir(exist_ok=True)
sortie = dossier_pop / f"{fichier.stem}.parquet"
pop.to_parquet(sortie, index=False)

# Relecture sans préciser les types, pour vérifier qu'ils sont conservés
relu = pd.read_parquet(sortie)
print("Lignes brutes :", n_brut, "| retenues :", len(pop))
print("Taille CSV     :", round(fichier.stat().st_size / 1e6, 1), "Mo")
print("Taille Parquet :", round(sortie.stat().st_size / 1e6, 1), "Mo")
print(relu[["coddep", "l_codinsee"]].dtypes)
print(relu["l_codinsee"].head(3))

Lignes brutes : 24793 | retenues : 8880
Taille CSV     : 16.2 Mo
Taille Parquet : 1.6 Mo
coddep        str
l_codinsee    str
dtype: object
0    48009
1    48095
2    48095
Name: l_codinsee, dtype: str


### Traiter l'ensemble des départements (hors DOM)

In [23]:
import time

types_texte = {"coddep": str, "l_codinsee": str}  # codes géographiques lus en texte
bilan = []            # une ligne par département : lignes brutes et retenues
debut = time.time()   # chronomètre

# Boucle sur les 93 départements
for fichier in fichiers:
    morceaux = []
    n_brut = 0
    # Lecture par blocs pour limiter la mémoire utilisée
    for bloc in pd.read_csv(fichier, sep="|", chunksize=200_000,
                            dtype=types_texte, low_memory=False):
        n_brut += len(bloc)
        morceaux.append(filtre_perimetre(bloc))
    pop = pd.concat(morceaux, ignore_index=True)
    # Un fichier Parquet par département
    pop.to_parquet(dossier_pop / f"{fichier.stem}.parquet", index=False)
    bilan.append({"fichier": fichier.stem, "brut": n_brut, "retenu": len(pop)})
    print(f"{fichier.stem} : {n_brut} -> {len(pop)}")

# Tableau récapitulatif, conservé pour la note de synthèse
bilan = pd.DataFrame(bilan)
bilan["part_retenue_%"] = (bilan["retenu"] / bilan["brut"] * 100).round(1)
bilan.to_csv(dossier.parent / "bilan_filtrage.csv", index=False)

print("\nDurée :", round((time.time() - debut) / 60, 1), "minutes")
print(bilan[["brut", "retenu"]].sum())

dvf_plus_d01 : 169939 -> 89034
dvf_plus_d02 : 114948 -> 66822
dvf_plus_d03 : 92823 -> 53896
dvf_plus_d04 : 52456 -> 30931
dvf_plus_d05 : 63921 -> 34872
dvf_plus_d06 : 410396 -> 281850
dvf_plus_d07 : 96967 -> 43796
dvf_plus_d08 : 63458 -> 32707
dvf_plus_d09 : 55299 -> 27272
dvf_plus_d10 : 82519 -> 44555
dvf_plus_d11 : 147346 -> 82112
dvf_plus_d12 : 87988 -> 40335
dvf_plus_d13 : 470197 -> 301023
dvf_plus_d14 : 216700 -> 134210
dvf_plus_d15 : 42914 -> 20356
dvf_plus_d16 : 109860 -> 54737
dvf_plus_d17 : 261139 -> 141319
dvf_plus_d18 : 81079 -> 47724
dvf_plus_d19 : 72705 -> 35488
dvf_plus_d21 : 158864 -> 90938
dvf_plus_d22 : 194122 -> 108618
dvf_plus_d23 : 49318 -> 20129
dvf_plus_d24 : 139106 -> 67915
dvf_plus_d25 : 137340 -> 75866
dvf_plus_d26 : 136994 -> 75787
dvf_plus_d27 : 132528 -> 84222
dvf_plus_d28 : 105726 -> 64592
dvf_plus_d29 : 272898 -> 163564
dvf_plus_d2a : 46559 -> 21704
dvf_plus_d2b : 48470 -> 22171
dvf_plus_d30 : 209369 -> 117651
dvf_plus_d31 : 377615 -> 225677
dvf_plus_d32 :

### Contrôle de la population obtenue

In [24]:
# Taille totale des fichiers Parquet produits
parquets = list(dossier_pop.glob("*.parquet"))
taille_go = sum(p.stat().st_size for p in parquets) / 1e9
print("Fichiers Parquet :", len(parquets))
print("Taille totale :", round(taille_go, 2), "Go")

# Zéro devant les codes : contrôle sur l'Ain (01), en ne lisant que 2 colonnes
d01 = pd.read_parquet(dossier_pop / "dvf_plus_d01.parquet",
                      columns=["coddep", "l_codinsee"])
print(d01.head(3))

# Départements où la part du résidentiel est la plus faible et la plus forte
print(bilan.sort_values("part_retenue_%").head(5))
print(bilan.sort_values("part_retenue_%").tail(5))

Fichiers Parquet : 93
Taille totale : 1.4 Go
  coddep l_codinsee
0     01      01378
1     01      01160
2     01      01022
         fichier   brut  retenu  part_retenue_%
48  dvf_plus_d48  24793    8880            35.8
43  dvf_plus_d43  77957   30440            39.0
21  dvf_plus_d23  49318   20129            40.8
46  dvf_plus_d46  59242   26369            44.5
67  dvf_plus_d70  65887   29379            44.6
         fichier    brut  retenu  part_retenue_%
5   dvf_plus_d06  410396  281850            68.7
91  dvf_plus_d94  257058  176783            68.8
75  dvf_plus_d78  289392  199780            69.0
88  dvf_plus_d91  250046  174024            69.6
72  dvf_plus_d75  498059  357999            71.9


### Résultat du filtrage

- 93 départements traités en 2,6 minutes.
- 16 375 738 mutations brutes → **9 680 495 ventes retenues (59,1 %)**, enregistrées en 93 fichiers Parquet (1,4 Go).
- Les codes géographiques restent en texte : le zéro initial est conservé (`01`, `01378`).
- La part du résidentiel varie du simple au double selon les départements : 35,8 % en Lozère, 71,9 % à Paris. C'est ce qui justifie de filtrer avant de tirer l'échantillon.

### Compter les ventes par strate

In [25]:
# Nombre de ventes par strate (département × année), sur toute la population
comptes = []
for p in sorted(parquets):
    # On ne lit que les 2 colonnes qui définissent la strate
    d = pd.read_parquet(p, columns=["coddep", "anneemut"])
    # value_counts compte chaque combinaison (département, année)
    comptes.append(d.value_counts().rename("N_h").reset_index())
strates = pd.concat(comptes, ignore_index=True)

print("Nombre de strates :", len(strates))
print(strates.groupby("anneemut")["N_h"].sum())
print(strates.sort_values("N_h").head(5))

Nombre de strates : 1116
anneemut
2014     592452
2015     673224
2016     718588
2017     819528
2018     818058
2019     904755
2020     881175
2021    1022405
2022     978012
2023     773771
2024     703296
2025     795231
Name: N_h, dtype: int64
    coddep  anneemut  N_h
587     48      2014  549
586     48      2016  551
585     48      2015  563
584     48      2017  673
583     48      2018  708


### Calculer combien tirer dans chaque strate

In [26]:
# Taux de tirage et plancher par strate
taux, m = 0.10, 300

app = strates[strates["anneemut"] <= 2023].copy()
N = app["N_h"].sum()

# Part proportionnelle (10 %), relevée au plancher, plafonnée à la taille de la strate
app["n_h"] = np.minimum(
    app["N_h"],
    np.maximum(m, np.round(app["N_h"] * taux)),
).astype(int)

app["w_h"] = app["N_h"] / app["n_h"]

print("Population 2014-2023 :", N)
print("Taille de l'échantillon :", app["n_h"].sum())
print("Strates relevées par le plancher :", (app["n_h"] > np.round(app["N_h"] * taux)).sum())
print(app.sort_values("w_h").head(3))
print(app.sort_values("w_h").tail(3))

Population 2014-2023 : 8181968
Taille de l'échantillon : 835288
Strates relevées par le plancher : 188
    coddep  anneemut  N_h  n_h       w_h
587     48      2014  549  300  1.830000
586     48      2016  551  300  1.836667
585     48      2015  563  300  1.876667
    coddep  anneemut   N_h  n_h        w_h
96      09      2021  3165  316  10.015823
51      05      2020  3165  316  10.015823
804     70      2021  3025  302  10.016556


### Plan de tirage

- Population d'apprentissage (2014-2023) : 8 181 968 ventes, réparties en 930 strates (93 départements × 10 années).
- Tirage d'**une vente sur dix** dans chaque strate (taux de 10 %), avec un plancher de 300 ventes par strate.
- 188 strates (20 %) sont relevées par le plancher. Ce sont celles de moins de 3 000 ventes, dans les départements peu peuplés.
- Taille de l'échantillon : **835 288 ventes**, dont 17 091 (2,0 %) ajoutées par le plancher.
- Poids `w_h` compris entre 1,83 (Lozère 2014) et 10,02. **L'échantillon n'est pas proportionnel : toute statistique ou tout modèle portant sur plusieurs départements ou plusieurs années doit utiliser `w_h`.**

### Tirer l'échantillon

In [27]:
# Générateur de nombres aléatoires, graine fixée pour la reproductibilité
rng = np.random.default_rng(42)
morceaux = []

for p in sorted(dossier_pop.glob("*.parquet")):
    d = pd.read_parquet(p)
    # Années d'apprentissage uniquement
    d = d[d["anneemut"] <= 2023]
    # On ajoute à chaque vente le n_h et le w_h de sa strate
    d = d.merge(app[["coddep", "anneemut", "n_h", "w_h"]], on=["coddep", "anneemut"])
    # Ticket de loterie : un nombre au hasard par vente
    d["alea"] = rng.random(len(d))
    # Rang du ticket dans sa strate (1 = plus petit numéro)
    d["rang"] = d.groupby(["coddep", "anneemut"])["alea"].rank(method="first")
    # On garde les n_h premiers rangs de chaque strate
    morceaux.append(d[d["rang"] <= d["n_h"]].drop(columns=["alea", "rang", "n_h"]))

ech = pd.concat(morceaux, ignore_index=True)

# Contrôles : taille totale et respect de n_h dans chaque strate
print("Taille de l'échantillon :", len(ech))
verif = ech.groupby(["coddep", "anneemut"]).size().rename("tire").reset_index()
verif = verif.merge(app[["coddep", "anneemut", "n_h"]], on=["coddep", "anneemut"])
print("Strates où le nombre tiré diffère de n_h :", (verif["tire"] != verif["n_h"]).sum())
print("idmutation uniques :", ech["idmutation"].is_unique)

Taille de l'échantillon : 835288
Strates où le nombre tiré diffère de n_h : 0
idmutation uniques : True


### Enregistrer l'échantillon et la liste des identifiants

In [28]:
# Chemins de sortie, dans data/
chemin_ech = dossier.parent / "dvf_plus_echantillon_apprentissage.parquet"
chemin_ids = dossier.parent / "ids_echantillon_apprentissage.csv"

# Échantillon complet (avec w_h) et liste des identifiants tirés
ech.to_parquet(chemin_ech, index=False)
ech[["idmutation"]].to_csv(chemin_ids, index=False)

# Taille des deux fichiers
print("Échantillon :", round(chemin_ech.stat().st_size / 1e6, 1), "Mo")
print("Identifiants :", round(chemin_ids.stat().st_size / 1e6, 1), "Mo")

Échantillon : 123.9 Mo
Identifiants : 6.9 Mo


### Choix du format d'enregistrement de l'échantillon

L'échantillon complet pèse 124 Mo, ce qui le rend lourd à partager et dépasse la limite de 100 Mo par fichier de GitHub. On teste deux leviers pour l'alléger, sans modifier le tirage :
- retirer les 11 colonnes non conservées à l'issue de l'audit (vides, constantes, redondantes ou inutiles pour le modèle) ;
- utiliser une compression plus forte (`zstd`) que celle par défaut (`snappy`).

In [29]:
# Colonnes vides, constantes, ou listes inutiles pour le modèle
a_retirer = ["idmutinvar", "idopendata", "codservch", "refdoc", "nbartcgi", "l_artcgi",
             "l_section", "l_idpar", "l_idparmut", "l_idlocmut", "l_dcnt"]
ech_leger = ech.drop(columns=a_retirer)

# Trois versions de test
essais = {
    "toutes colonnes, zstd": (ech, "zstd"),
    "colonnes allégées, snappy": (ech_leger, "snappy"),
    "colonnes allégées, zstd": (ech_leger, "zstd"),
}
for nom, (tableau, compression) in essais.items():
    chemin = dossier.parent / "test_taille.parquet"
    tableau.to_parquet(chemin, index=False, compression=compression)
    print(f"{nom} : {round(chemin.stat().st_size / 1e6, 1)} Mo")
    chemin.unlink()   # supprime le fichier de test

toutes colonnes, zstd : 77.3 Mo
colonnes allégées, snappy : 33.4 Mo
colonnes allégées, zstd : 26.0 Mo


**Résultat :**

Le retrait des colonnes est le levier principal (−73 %) : les listes de texte et les identifiants de 32 caractères représentaient l'essentiel du poids. 
La version allégée avec la compression par défaut (snappy) est retenue : 33 Mo suffisent pour le partage, et zstd n'apporterait que 7 Mo de moins.

In [30]:
# Échantillon allégé : colonnes non conservées retirées (elles restent dans la population)
ech_leger = ech.drop(columns=a_retirer)

# On remplace le fichier de 124 Mo par la version allégée (compression par défaut)
ech_leger.to_parquet(chemin_ech, index=False)

print("Dimensions :", ech_leger.shape)
print("Taille :", round(chemin_ech.stat().st_size / 1e6, 1), "Mo")

Dimensions : (835288, 53)
Taille : 33.4 Mo


### Échantillon enregistré

- `data/dvf_plus_echantillon_apprentissage.parquet` : 835 288 ventes, 53 colonnes, 33 Mo. À partager par le drive (hors GitHub).
- `data/ids_echantillon_apprentissage.csv` : liste des `idmutation` tirés (6,9 Mo), versionnée sur GitHub.
- Colonnes non conservées à l'issue de l'audit : `idmutinvar`, `idopendata`, `codservch`, `refdoc`, `nbartcgi`, `l_artcgi`, `l_section`, `l_idpar`, `l_idparmut`, `l_idlocmut`, `l_dcnt`. Elles restent disponibles dans la population (`data/dvf_plus_residentiel/`) et peuvent être rattachées par `idmutation`.
- Clés de jointure conservées : `idmutation` (identifiant), `anneemut` et `moismut` (taux mensuels), `l_codinsee` (commune), `geompar_x` et `geompar_y` (rattachement éventuel à l'IRIS).
- **L'échantillon n'est pas proportionnel** : toute statistique ou tout modèle portant sur plusieurs départements ou plusieurs années doit utiliser la colonne `w_h`.

### Vérifier que l'échantillon pondérée est représentatif

In [31]:
def mediane_ponderee(valeurs, poids):
    """Médiane où chaque valeur compte autant que son poids."""
    v = np.asarray(valeurs)
    w = np.asarray(poids)
    ordre = np.argsort(v)                 # rang des valeurs, de la plus petite à la plus grande
    v, w = v[ordre], w[ordre]
    cumul = np.cumsum(w)                  # poids additionnés au fur et à mesure
    return v[np.searchsorted(cumul, cumul[-1] / 2)]   # valeur où l'on atteint la moitié du poids total

cols = ["anneemut", "libtypbien", "valeurfonc", "sbati"]

# Population complète 2014-2023, en ne lisant que les colonnes utiles
popu = pd.concat([pd.read_parquet(p, columns=cols) for p in sorted(dossier_pop.glob("*.parquet"))],
                 ignore_index=True)
popu = popu[popu["anneemut"] <= 2023]

# Échantillon relu depuis le fichier enregistré (vérifie au passage le fichier lui-même)
e = pd.read_parquet(chemin_ech, columns=cols + ["w_h"])

# Prix au m², en écartant les valeurs foncières manquantes
popu = popu.dropna(subset=["valeurfonc"])
popu["prix_m2"] = popu["valeurfonc"] / popu["sbati"]
e = e.dropna(subset=["valeurfonc"])
e["prix_m2"] = e["valeurfonc"] / e["sbati"]

# Comparaison des trois versions
comparaison = pd.DataFrame({
    "population": [
        (popu["libtypbien"] == "UNE MAISON").mean() * 100,
        popu["prix_m2"].median(),
    ],
    "échantillon sans poids": [
        (e["libtypbien"] == "UNE MAISON").mean() * 100,
        e["prix_m2"].median(),
    ],
    "échantillon avec poids": [
        np.average(e["libtypbien"] == "UNE MAISON", weights=e["w_h"]) * 100,
        mediane_ponderee(e["prix_m2"], e["w_h"]),
    ],
}, index=["part de maisons (%)", "prix médian au m² (€)"]).round(1)

print(comparaison)

                       population  échantillon sans poids  \
part de maisons (%)          57.3                    57.7   
prix médian au m² (€)      2250.0                  2222.2   

                       échantillon avec poids  
part de maisons (%)                      57.3  
prix médian au m² (€)                  2247.2  


L'échantillon pondéré reproduit la population : part de maisons identique, prix médian à 0,1 % près. Sans poids, l'échantillon contient plus de maisons et sous-estime le prix médian de 1,2 %, à cause de la surreprésentation des départements peu peuplés. 
Le choix est fait d'adopter les poids `w_h` pour toute statistique nationale.

## Conclusion de la partie 1

### Ce que nous avons fait

1. **Filtrer les données.** Dans les 93 fichiers de France métropolitaine, nous avons gardé uniquement les ventes d'une seule maison ou d'un seul appartement. Résultat : 9 680 495 ventes sur 16 375 738 mutations (59 %).
2. **Tirer un échantillon.** Sur les années 2014 à 2023, nous avons tiré au hasard une vente sur dix dans chaque département et chaque année, avec au moins 300 ventes à chaque fois. Résultat : 835 288 ventes.
3. **Alléger le fichier.** Nous avons retiré les 11 colonnes inutiles repérées pendant l'audit. Le fichier passe de 124 Mo à 33 Mo.

### Pourquoi

- L'échantillon de l'audit contenait trop peu de ventes par département et par année.
- La part des ventes de logements varie beaucoup selon les départements : 36 % en Lozère, 72 % à Paris. Il fallait donc filtrer avant de tirer.
- Le minimum de 300 ventes permet d'avoir assez de données dans les petits départements.

### Résultat

Grâce aux poids (colonne `w_h`), l'échantillon donne les mêmes résultats que l'ensemble des ventes : 57,3 % de maisons et un prix médian d'environ 2 250 €/m².

### À retenir pour la suite

- Toujours utiliser la colonne `w_h` pour les calculs sur toute la France.
- Calculer les prix par commune sur l'ensemble des ventes, pas sur l'échantillon.
- Les échantillons 2024 et 2025 restent à tirer.

# Partie 2 — Table cible : prix médian au m² par commune et par année

Cette table sert de cible commune à toute l'équipe : chacun l'utilise pour croiser sa source avec les prix.

- Elle est calculée sur **toutes les ventes** retenues (9,7 millions), pas sur l'échantillon. Ainsi, chaque commune a le plus de ventes possible.
- **Une ligne = une commune × une année.** Pour chaque type de bien (maison, appartement), on donne le nombre de ventes et le prix médian au m².
- On utilise la médiane plutôt que la moyenne, car quelques ventes très chères suffisent à tirer la moyenne vers le haut.

### Charger la population (colonnes utiles seulement)

In [32]:
import pandas as pd

# Seulement les colonnes utiles : la population complète tient alors en mémoire
colonnes = ["idmutation", "anneemut", "l_codinsee", "libtypbien", "valeurfonc", "sbati"]

pop = pd.read_parquet(dossier_pop, columns=colonnes)

print(pop.shape)
pop.head()

(9680495, 6)


,idmutation,anneemut,l_codinsee,libtypbien,valeurfonc,sbati
0,16,2014,01378,UNE MAISON,222300.0,88.0
1,476,2014,01160,UNE MAISON,1084000.0,212.0
2,8975,2014,01022,UNE MAISON,139000.0,90.0
3,9113,2014,01210,UN APPARTEMENT,75700.0,41.0
4,9498,2014,01185,UNE MAISON,58000.0,39.0


### Calculer le prix au m² et retirer les valeurs aberrantes

On applique les trois règles d'exclusion fixées pendant l'EDA :
- la valeur foncière doit être renseignée et positive ;
- le prix au m² doit être compris entre 50 € et 50 000 €.

In [33]:
# Prix au m² de chaque vente
pop["prix_m2"] = pop["valeurfonc"] / pop["sbati"]

# Les trois règles d'exclusion de l'EDA
avant = len(pop)
pop = pop[
    pop["valeurfonc"].notna()
    & (pop["valeurfonc"] > 0)
    & (pop["prix_m2"] >= 50)
    & (pop["prix_m2"] <= 50000)
]
apres = len(pop)
print(f"Avant : {avant:,} | Après : {apres:,} | Exclues : {avant - apres:,} ({(avant - apres) / avant:.2%})")

# Type de bien en version courte, pour les noms de colonnes
pop["type_bien"] = pop["libtypbien"].map({"UNE MAISON": "maison", "UN APPARTEMENT": "appart"})
print(pop["type_bien"].value_counts(dropna=False))

Avant : 9,680,495 | Après : 9,655,421 | Exclues : 25,074 (0.26%)
type_bien
maison    5523372
appart    4132049
Name: count, dtype: int64


Ces règles retirent 25 074 ventes, soit 0,26 % de la population. Il reste 5,5 millions de maisons et 4,1 millions d'appartements.

### Agréger par commune et par année

In [34]:
# Une ligne par commune × année × type, puis le type passe en colonnes
cible = (
    pop.groupby(["l_codinsee", "anneemut", "type_bien"])["prix_m2"]
    .agg(n="count", prix_m2_med="median")
    .unstack("type_bien")
)
# Noms de colonnes : ("n", "maison") devient "n_maison"
cible.columns = [f"{stat}_{typ}" for stat, typ in cible.columns]
cible = cible.reset_index().rename(columns={"l_codinsee": "code_commune", "anneemut": "annee"})

# Une commune sans vente d'un type : n = 0 (le prix reste vide)
for col in ["n_maison", "n_appart"]:
    cible[col] = cible[col].fillna(0).astype(int)

# Prix arrondis à l'euro
cible[["prix_m2_med_appart", "prix_m2_med_maison"]] = cible[["prix_m2_med_appart", "prix_m2_med_maison"]].round(0)

# Les six colonnes, dans l'ordre choisi
cible = cible[["code_commune", "annee", "n_appart", "n_maison", "prix_m2_med_appart", "prix_m2_med_maison"]]

print(cible.shape)
print("Total des ventes :", cible["n_appart"].sum() + cible["n_maison"].sum())   # doit valoir 9 655 421
cible.head()

(367779, 6)
Total des ventes : 9655421


,code_commune,annee,n_appart,n_maison,prix_m2_med_appart,prix_m2_med_maison
0,01001,2014,0,10,NaN,1893.0
1,01001,2015,0,10,NaN,1782.0
2,01001,2016,0,7,NaN,1807.0
3,01001,2017,0,14,NaN,1827.0
4,01001,2018,0,9,NaN,1933.0


In [35]:
# Enregistrement de la table cible (commune × année × type), en Parquet et en CSV
dossier_cible = dossier.parent / "dataset_cible"
dossier_cible.mkdir(exist_ok=True)

cible.to_parquet(dossier_cible / "cible_commune_annee_type.parquet", index=False)
cible.to_csv(dossier_cible / "cible_commune_annee_type.csv", index=False)

# Contrôle : relecture du Parquet, code commune toujours en texte
relu = pd.read_parquet(dossier_cible / "cible_commune_annee_type.parquet")
print(relu.shape, relu["code_commune"].dtype, relu["code_commune"].head(2).tolist())

(367779, 6) str ['01001', '01001']


### Résultat

- **367 779 lignes** (commune × année) et 6 colonnes.
- Contrôle : la somme des ventes vaut 9 655 421, soit exactement le nombre de ventes après exclusion. Aucune vente n'est perdue.
- Quand une commune n'a aucune vente d'un type de bien une année donnée, le nombre vaut 0 et le prix reste vide.
- La table couvre 2014-2025. **L'année 2025 est réservée au test** : il faut l'exclure dans les analyses exploratoires.

- Table enregistrée dans `data/dataset_cible/` : `cible_commune_annee_type.parquet` (à privilégier, le code commune y reste en texte) et `cible_commune_annee_type.csv`.

# Partie 3 — Filosofi : revenus des ménages par commune

Filosofi est une source de l'Insee. Elle décrit le **revenu disponible** des ménages (revenus après impôts et prestations sociales) à partir des données fiscales. On utilise les fichiers par commune, de 2014 à 2021 : un fichier par année, appelée « millésime ».

Les huit fichiers ne sont pas tous au même format :
- 2014 à 2017 : fichiers Excel, dont les noms de colonnes sont à la 6e ligne ;
- 2018 à 2021 : fichiers CSV séparés par des points-virgules ;
- dans tous les cas, les noms de colonnes se terminent par l'année (`Q214` = médiane de 2014).

**Une ligne = une commune × une année.**

### Inspecter les huit fichiers

In [36]:
# Imports communs à tout le notebook
from pathlib import Path
import re
import time
import numpy as np
import pandas as pd
dossier_filo = Path("/home/naude/formation/projet_compagnon/data/Filo_disp_com")

for f in sorted(dossier_filo.glob("FILO*")):
    print("==", f.name)
    if f.suffix in (".xls", ".xlsx"):
        haut = pd.read_excel(f, sheet_name="ENSEMBLE", header=None, nrows=6)
        print("   ", haut.iloc[2, 0])                    # mise en ligne et géographie
        print("    colonnes :", list(haut.iloc[5, :8]))  # ligne des codes
    else:
        with open(f, encoding="utf-8", errors="replace") as fh:
            print("    colonnes :", fh.readline()[:150])

== FILO_DISP_COM_2014.xls
    Mise en ligne le 25/04/2019       Géographie au 01/01/2015
    colonnes : ['CODGEO', 'LIBGEO', 'NBMEN14', 'NBPERS14', 'NBUC14', 'Q114', 'Q214', 'Q314']
== FILO_DISP_COM_2015.xls
    Mise en ligne le 25/04/2019       Géographie au 01/01/2016
    colonnes : ['CODGEO', 'LIBGEO', 'NBMEN15', 'NBPERS15', 'NBUC15', 'Q115', 'Q215', 'Q315']
== FILO_DISP_COM_2016.xls
    Mise en ligne le 26/07/2019       Géographie au 01/01/2017
    colonnes : ['CODGEO', 'LIBGEO', 'NBMEN16', 'NBPERS16', 'NBUC16', 'Q116', 'Q216', 'Q316']
== FILO_DISP_COM_2017.xlsx
    Mise en ligne le 23/01/2020       Géographie au 01/01/2018
    colonnes : ['CODGEO', 'LIBGEO', 'NBMEN17', 'NBPERS17', 'NBUC17', 'Q117', 'Q217', 'Q317']
== FILO_DISP_COM_2018.csv
    colonnes : CODGEO;NBMEN18;NBPERS18;NBUC18;Q118;Q218;Q318;Q3_Q1;D118;D218;D318;D418;D618;D718;D818;D918;RD;S80S2018;GI18;PACT18;PTSA18;PCHO18;PBEN18;PPEN18;PPAT18
== FILO_DISP_COM_2019.csv
    colonnes : CODGEO;NBMEN19;NBPERS19;NBUC19;Q119;Q2

Chaque millésime est publié dans la géographie de l'année suivante (« Géographie au 01/01/2015 » pour 2014). Les communes ayant fusionné entre deux années peuvent donc avoir changé de code.

### Assembler les huit millésimes

On garde **toutes les colonnes présentes les huit années**, pour pouvoir suivre chaque indicateur dans le temps. Le tri entre colonnes utiles et inutiles se fera après l'audit.

La cellule procède en trois temps :
1. lire chaque fichier et retirer l'année à la fin des noms de colonnes (`Q214` devient `Q2`) ;
2. repérer les colonnes communes aux huit années ;
3. garder ces colonnes et empiler les huit tableaux, avec une colonne `annee`.
4. convertir toutes les colonnes en nombres, sauf le code commune. En 2020 et 2021, les décimales sont écrites avec une virgule (`0,225`) et les valeurs soumises au secret statistique sont notées `s` : la virgule est remplacée par un point, et `s` devient une valeur vide.

Les colonnes écartées sont affichées pour chaque année. Cela permet de vérifier qu'aucune colonne n'est perdue à cause d'un nom écrit différemment selon les années.

In [37]:
import re

def en_nombre(df, cols):
    """Convertit des colonnes en nombres (virgule -> point, 's' -> vide)."""
    return df[cols].apply(lambda s: pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False),
                                                  errors="coerce"))

# 1. Lire et renommer chaque fichier, sans filtrer
tableaux = []
for f in sorted(dossier_filo.glob("FILO*DISP_COM*")):
    annee = int(re.search(r"20\d\d", f.name).group())
    aa = str(annee)[2:]                       # "14", "15", ...

    if f.suffix in (".xls", ".xlsx"):
        df = pd.read_excel(f, sheet_name="ENSEMBLE", header=5, dtype={"CODGEO": str})
    else:
        df = pd.read_csv(f, sep=";", dtype={"CODGEO": str})

    # On retire l'année à la fin des noms : "Q214" -> "Q2"
    df.columns = [c[:-2] if c.endswith(aa) and c != "CODGEO" else c for c in df.columns]
    df["annee"] = annee

    # 2014 : PACT n'existe pas. PTSAC (2014) = PTSA + PCHO (2015+), donc PACT = PTSAC + PBEN
    if annee == 2014:
        x = en_nombre(df, ["PTSAC", "PBEN"])
        df["PACT"] = x["PTSAC"] + x["PBEN"]

    tableaux.append(df)

# 2. Colonnes présentes les 8 années (intersection)
communes = set(tableaux[0].columns)
for df in tableaux[1:]:
    communes = communes & set(df.columns)
communes = ["CODGEO", "annee"] + sorted(communes - {"CODGEO", "annee"})
print(len(communes), "colonnes communes :", communes)

# Contrôle : ce que chaque année perd
for df in tableaux:
    ecartees = sorted(set(df.columns) - set(communes))
    print(df["annee"].iloc[0], "|", len(ecartees), "écartées :", ecartees[:10])

# 3. Garder les colonnes communes et empiler
filo = pd.concat([df[communes] for df in tableaux], ignore_index=True)

# 4. Tout en nombres, sauf le code commune
cols_num = [c for c in filo.columns if c != "CODGEO"]
filo[cols_num] = en_nombre(filo, cols_num)

print(filo.shape)
print(filo.dtypes.value_counts())
filo.head()

/tmp/ipykernel_6793/2792702136.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["annee"] = annee
/tmp/ipykernel_6793/2792702136.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["annee"] = annee


29 colonnes communes : ['CODGEO', 'annee', 'D1', 'D2', 'D3', 'D4', 'D6', 'D7', 'D8', 'D9', 'GI', 'NBMEN', 'NBPERS', 'NBUC', 'PACT', 'PBEN', 'PIMPOT', 'PPAT', 'PPEN', 'PPFAM', 'PPLOGT', 'PPMINI', 'PPSOC', 'Q1', 'Q2', 'Q3', 'Q3_Q1', 'RD', 'S80S20']
2014 | 2 écartées : ['LIBGEO', 'PTSAC']
2015 | 3 écartées : ['LIBGEO', 'PCHO', 'PTSA']
2016 | 3 écartées : ['LIBGEO', 'PCHO', 'PTSA']
2017 | 3 écartées : ['LIBGEO', 'PCHO', 'PTSA']
2018 | 704 écartées : ['AGE1D1', 'AGE1D2', 'AGE1D3', 'AGE1D4', 'AGE1D6', 'AGE1D7', 'AGE1D8', 'AGE1D9', 'AGE1GI', 'AGE1PACT']
2019 | 704 écartées : ['AGE1D1', 'AGE1D2', 'AGE1D3', 'AGE1D4', 'AGE1D6', 'AGE1D7', 'AGE1D8', 'AGE1D9', 'AGE1GI', 'AGE1PACT']
2020 | 704 écartées : ['AGE1D1', 'AGE1D2', 'AGE1D3', 'AGE1D4', 'AGE1D6', 'AGE1D7', 'AGE1D8', 'AGE1D9', 'AGE1GI', 'AGE1PACT']
2021 | 704 écartées : ['AGE1D1', 'AGE1D2', 'AGE1D3', 'AGE1D4', 'AGE1D6', 'AGE1D7', 'AGE1D8', 'AGE1D9', 'AGE1GI', 'AGE1PACT']
(261319, 29)
float64    27
str         1
int64       1
Name: count, dtyp

,CODGEO,annee,D1,D2,D3,D4,D6,D7,D8,D9,...,PPFAM,PPLOGT,PPMINI,PPSOC,Q1,Q2,Q3,Q3_Q1,RD,S80S20
0,01001,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,21576.666667,NaN,NaN,NaN,NaN
1,01002,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,21672.857143,NaN,NaN,NaN,NaN
2,01004,2014,10545.666667,13184.666667,15507.333333,17564.0,21840.0,24469.2,27991.333333,33376.5,...,2.8,1.9,1.8,6.5,14340.0,19756.111111,25954.0,11614.0,3.164949,3.792281
3,01005,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,23204.800000,NaN,NaN,NaN,NaN
4,01006,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,22157.500000,NaN,NaN,NaN,NaN


### Contrôler la couverture et les doublons

In [38]:
couverture = filo.groupby("annee").agg(
    lignes=("CODGEO", "size"),
    mediane_remplie=("Q2", "count"),     # count ignore les valeurs vides
    gini_rempli=("GI", "count"),
    d1_rempli=("D1", "count"),
)
print(couverture)

# Une commune doit apparaître une seule fois par année
print("Doublons commune × année :", filo.duplicated(["CODGEO", "annee"]).sum())

# Paris, Lyon, Marseille : commune entière et arrondissements en même temps ?
print(filo.loc[filo["CODGEO"].str.match(r"^(75056|75101|69123|69381|13055|13201)$"), ["CODGEO", "annee"]]
      .groupby("CODGEO").size())

       lignes  mediane_remplie  gini_rempli  d1_rempli
annee                                                 
2014    32974            32974         5222       5222
2015    32252            32252         5257       5257
2016    31789            31789         5279       5279
2017    31745            31745         5304       5304
2018    31330            31330         5307       5307
2019    31361            31361         5336       5336
2020    34939            31336         5351       5351
2021    34929            31325         5346       5346
Doublons commune × année : 0
CODGEO
13055    8
13201    8
69123    8
69381    8
75056    8
75101    8
dtype: int64


**Lecture du tableau :**
- La médiane du niveau de vie est renseignée pour 31 300 à 33 000 communes selon les années, sur environ 35 000.
- En 2020 et 2021, le fichier compte environ 3 600 lignes de plus, mais sans médiane : les communes soumises au secret statistique y sont présentes avec des valeurs vides, alors qu'avant 2020 elles étaient absentes. Pour mesurer la couverture, on compte donc les médianes remplies, pas les lignes.
- D1, D9, Gini et les parts de revenus ne sont publiés que pour environ 5 300 communes chaque année (secret statistique : trop peu de ménages).
- Aucun doublon commune × année.
- Paris, Lyon et Marseille apparaissent à la fois comme commune entière (75056) et par arrondissement (75101…). Il faudra n'en garder qu'un seul niveau au moment de la jointure, pour ne pas compter deux fois.

### Renommer et enregistrer

In [39]:
filo = filo.rename(columns={"CODGEO": "code_commune", "Q2": "niveau_vie_median"})
filo.to_parquet("../data/filosofi_commune_2014_2021.parquet", index=False)
print(filo.shape)

(261319, 29)


### Résultat de la partie 3

- `data/filosofi_commune_2014_2021.parquet` : 261 319 lignes (commune × année), 29 colonnes, toutes numériques sauf le code commune.
- On garde toutes les colonnes présentes les huit années. La part des revenus d'activité (`PACT`) a été reconstruite pour 2014.
- Colonnes renommées pour la jointure avec la table cible : `code_commune`, `niveau_vie_median`.

**Points à traiter pendant l'audit de Filosofi :**
- choisir les colonnes à garder pour l'analyse, parmi les 29 ;
- choisir entre commune entière et arrondissements pour Paris, Lyon et Marseille ;
- vérifier les codes communes qui changent entre millésimes (fusions de communes) ;
- Filosofi s'arrête en 2021 alors que DVF+ va jusqu'en 2025 : décider comment traiter les années sans revenu.

# Partie 5 Test de rattachement des communes de DVF+ aux bassins de vie 2022

**Objectif :** vérifier si chaque code commune de la table cible DVF+ trouve son bassin de vie dans le fichier Insee de composition communale des bassins de vie 2022.

**Données :**
- `data/dataset_cible/cible_commune_annee_type.parquet` : table cible DVF+ (une ligne par commune × année, nombre de ventes et prix médian au m² par type de bien). Les codes commune sont ceux de DVF+ (`l_codinsee`), donc dans la géographie de l'année de chaque vente.
- `data/BV2022_2022_2026.xlsx`, onglet `Composition_communale` : chaque commune et son bassin de vie, **en géographie au 01/01/2026** (34 875 communes).

**Méthode :** jointure à gauche de la table cible sur le fichier des bassins, puis comptage des lignes et des ventes sans bassin.

In [1]:
import pandas as pd

# Table cible DVF+ : une ligne par commune × année
cible = pd.read_parquet("../data/dataset_cible/cible_commune_annee_type.parquet")

# Composition communale des bassins de vie 2022 (géographie 2026)
# Les 5 premières lignes sont des titres : l'en-tête est sur la ligne 5.
# dtype=str garde le zéro initial des codes (01001 et non 1001).
bv = pd.read_excel("../data/BV2022_2022_2026.xlsx", sheet_name="Composition_communale",
                   skiprows=5, dtype=str)

# Jointure à gauche : on garde toutes les lignes de la cible.
# indicator=True ajoute une colonne _merge :
#   "both"      -> la commune a trouvé son bassin
#   "left_only" -> la commune n'existe pas dans le fichier des bassins
test = cible.merge(bv[["CODGEO", "BV2022"]], left_on="code_commune", right_on="CODGEO",
                   how="left", indicator=True)

# Nombre de ventes de chaque ligne (appartements + maisons)
ventes = test["n_appart"] + test["n_maison"]

print(test["_merge"].value_counts())
print("Ventes sans bassin :", ventes[test["_merge"] == "left_only"].sum(), "sur", ventes.sum())

# Codes sans bassin, regroupés par leurs 3 premiers caractères
# pour repérer d'où ils viennent (département, arrondissements...)
print(test.loc[test["_merge"] == "left_only", "code_commune"].str[:3].value_counts().head(10))

_merge
both          361530
left_only       6249
right_only         0
Name: count, dtype: int64
Ventes sans bassin : 620813 sur 9655421
code_commune
751    240
132    192
491    165
490    160
492    136
493    111
693    108
611     87
146     86
792     83
Name: count, dtype: int64


### Répartition par cause

In [2]:
# On isole les lignes sans bassin
perdues = test[test["_merge"] == "left_only"].copy()
perdues["ventes"] = perdues["n_appart"] + perdues["n_maison"]

# Deux causes possibles :
# - arrondissements de Paris (751..), Lyon (6938.) et Marseille (132..) :
#   DVF+ les utilise, le fichier Insee ne connaît que la ville entière ;
# - tout le reste : anciens codes de communes fusionnées depuis 2014.
perdues["cause"] = perdues["code_commune"].str[:3].isin(["751", "132", "693"]).map(
    {True: "arrondissement", False: "fusion ou autre"})

perdues.groupby("cause").agg(lignes=("code_commune", "size"),
                             communes=("code_commune", "nunique"),
                             ventes=("ventes", "sum"))

,lignes,communes,ventes
cause,,,
arrondissement,540,45,582153
fusion ou autre,5709,1726,38660


### DVF+ utilise t il les arrondissements ou ville entière? 

In [3]:
codes = cible["code_commune"]

# Nombre de codes d'arrondissement distincts (attendu : 20 + 9 + 16 = 45)
print("Arrondissements :", codes[codes.str.match(r"^(751|6938|132)")].nunique())

# Codes « ville entière » : une liste vide veut dire que DVF+ n'utilise que les arrondissements
print("Villes entières :", codes[codes.isin(["75056", "69123", "13055"])].unique())

Arrondissements : 45
Villes entières : <ArrowStringArray>
[]
Length: 0, dtype: str


**Constat :**

- 6 249 lignes commune × année sur 367 779 ne trouvent pas de bassin, soit **620 813 ventes sur 9 655 421 (6,4 %)**.
- **Arrondissements** : 45 codes (751.., 6938., 132..), présents les 12 années, soit **582 153 ventes (6,0 %)**. DVF+ code Paris, Lyon et Marseille par arrondissement ; le fichier Insee ne connaît que 75056, 69123 et 13055.
- **Communes fusionnées** : 1 726 anciens codes, soit **38 660 ventes (0,4 %)**, surtout dans le Maine-et-Loire (49), l'Orne (61), le Calvados (14) et les Deux-Sèvres (79), où beaucoup de communes nouvelles ont été créées depuis 2015.

**Impact projet :**

Sans correction, Paris, Lyon et Marseille disparaissent entièrement de la maille bassin de vie. Les fusions pèsent peu en ventes, mais concernent beaucoup de codes : elles toucheront aussi les jointures des autres sources (Filosofi, délinquance, ménages), qui ont chacune leur propre géographie.

**Action envisagée :**

Construire une table de correspondance unique pour toute l'équipe : tout code commune passé ou actuel → code commune 2026 → bassin de vie 2022.
- fusions : table de passage Insee `table_passage_geo2003_geo2026` ;
- arrondissements : règle 751xx → 75056, 6938x → 69123, 132xx → 13055.

**Critère de validation :** relancer ce test avec la table de correspondance. Il ne doit plus rester de vente sans bassin, ou seulement des cas identifiés et expliqués.

**Incertitudes / limites :**

- Le nombre de codes perdus dépend de la géographie du fichier des bassins : un premier essai, probablement en géographie 2024, en donnait 1 666 (31 820 ventes). Les fusions de 2025 et 2026 expliquent l'écart.
- Une médiane ne se moyenne pas : le prix médian par bassin × année devra être recalculé à partir des ventes, et non à partir des médianes communales.

## Test avec la table de correspondance des bassins de vie

**Objectif :** vérifier que la table `bv_liaison_2022.csv` rattache chaque commune à un bassin de vie 2022, sans perte ni duplication, pour les  sources : 
- les ventes DVF+, 
- la table cible agrégée 
- la délinquance.

**Deux clés selon la géographie de la source :**
- DVF+ garde le code de la commune à la date de la vente : jointure sur `CODGEO_INI` ;
- la délinquance est en géographie 2026 : jointure sur `CODGEO_2026`.

**Règles appliquées à la table :**
- commune scindée (un code initial, plusieurs codes 2026) : on garde la ligne où le code initial pointe vers lui-même ;
- 52427 (seul code scindé sans ligne identité) : on garde la ligne vers La Porte du Der (52331).

In [1]:
from pathlib import Path
import pandas as pd

dossier_data = Path("/home/naude/formation/projet_compagnon/data")

# Table de correspondance : plusieurs lignes par code initial en cas de scission
liaison = pd.read_csv(dossier_data / "dataset_cible" / "bv_liaison_2022.csv",
                      dtype=str, index_col=0)

# Clé 1 (sources en géographie d'origine, comme DVF+) : une ligne par code initial
dup = liaison["CODGEO_INI"].duplicated(keep=False)                  # codes scindés
identite = liaison["CODGEO_INI"] == liaison["CODGEO_2026"]          # le code pointe vers lui-même
cas_52427 = (liaison["CODGEO_INI"] == "52427") & (liaison["CODGEO_2026"] == "52331")
liaison_ini = liaison.loc[~dup | identite | cas_52427, ["CODGEO_INI", "BV2022"]]

# Clé 2 (sources en géographie 2026, comme la délinquance) : une ligne par code 2026
liaison_2026 = liaison[["CODGEO_2026", "BV2022"]].drop_duplicates()

print("Lignes du fichier :", len(liaison))
print("Clé CODGEO_INI  : unique =", liaison_ini["CODGEO_INI"].is_unique, "|", len(liaison_ini), "codes")
print("Clé CODGEO_2026 : unique =", liaison_2026["CODGEO_2026"].is_unique, "|", len(liaison_2026), "codes")

Lignes du fichier : 36788
Clé CODGEO_INI  : unique = True | 36759 codes
Clé CODGEO_2026 : unique = True | 34875 codes


In [2]:
def tester_rattachement(df, col_code, table_cle, col_cle, col_poids=None):
    """Rattache df aux bassins de vie et affiche ce qui ne se rattache pas.

    df        : la source à tester
    col_code  : sa colonne code commune
    table_cle : liaison_ini ou liaison_2026
    col_cle   : "CODGEO_INI" ou "CODGEO_2026"
    col_poids : colonne qui compte les ventes d'une ligne (table agrégée) ; sinon 1 ligne = 1
    """
    # validate : erreur si un code apparaît deux fois dans la table de liaison
    res = df.merge(table_cle, left_on=col_code, right_on=col_cle,
                   how="left", validate="many_to_one")
    sans_bv = res["BV2022"].isna()

    print("Lignes avant / après :", len(df), "/", len(res))
    print("Lignes sans bassin   :", sans_bv.sum())
    if col_poids is not None:
        print("Ventes sans bassin   :", res.loc[sans_bv, col_poids].sum(),
              "sur", res[col_poids].sum())

    # Les codes qui ne trouvent pas de bassin, du plus fréquent au moins fréquent
    return res.loc[sans_bv, col_code].value_counts()


# Ventes DVF+ : population complète, 3 colonnes seulement
ventes = pd.read_parquet(dossier_data / "dvf_plus_residentiel",
                         columns=["idmutation", "anneemut", "l_codinsee"])

codes_perdus_ventes = tester_rattachement(ventes, "l_codinsee", liaison_ini, "CODGEO_INI")
codes_perdus_ventes.head(20)

Lignes avant / après : 9680495 / 9680495
Lignes sans bassin   : 0


Series([], Name: count, dtype: int64)

In [3]:
# Table cible : une ligne par commune × année
# En CSV, dtype=str est indispensable : sinon "01001" serait lu comme le nombre 1001
cible = pd.read_csv(dossier_data / "dataset_cible" / "cible_commune_annee_type.csv",
                    dtype={"code_commune": str})

# Nombre de ventes de chaque ligne, pour compter en ventes et pas seulement en lignes
cible["n_ventes"] = cible["n_appart"] + cible["n_maison"]

codes_perdus_cible = tester_rattachement(cible, "code_commune", liaison_ini, "CODGEO_INI",
                                         col_poids="n_ventes")
codes_perdus_cible.head(20)

Lignes avant / après : 367779 / 367779
Lignes sans bassin   : 0
Ventes sans bassin   : 0 sur 9655421


Series([], Name: count, dtype: int64)

In [4]:
delinq = pd.read_parquet(dossier_data / "dataset_delinquance_communale.parquet")

print(delinq.shape)
print(delinq.dtypes)
delinq.head()

(5238000, 13)
CODGEO_2026                    str
annee                        int32
indicateur                category
unite_de_compte           category
nombre                     float64
taux_pour_mille            float64
est_diffuse                    str
insee_pop                    int32
insee_pop_millesime          int32
insee_log                    int32
insee_log_millesime          int32
complement_info_nombre     float64
complement_info_taux       float64
dtype: object


,CODGEO_2026,annee,indicateur,unite_de_compte,nombre,taux_pour_mille,est_diffuse,insee_pop,insee_pop_millesime,insee_log,insee_log_millesime,complement_info_nombre,complement_info_taux
0,01001,2016,Violences physiques intrafamiliales,Victime,0.0,0.0,diff,767,2016,348,2016,NaN,NaN
1,01001,2016,Violences physiques hors cadre familial,Victime,NaN,NaN,ndiff,767,2016,348,2016,1.362069,0.959839
2,01001,2016,Violences sexuelles,Victime,0.0,0.0,diff,767,2016,348,2016,NaN,NaN
3,01001,2016,Vols avec armes,Infraction,0.0,0.0,diff,767,2016,348,2016,NaN,NaN
4,01001,2016,Vols violents sans arme,Infraction,0.0,0.0,diff,767,2016,348,2016,NaN,NaN


In [5]:
# Contrôles de structure
print("Codes distincts :", delinq["CODGEO_2026"].nunique())
print("Années :", sorted(delinq["annee"].unique()))
print("Indicateurs :", delinq["indicateur"].nunique())

# Paris, Lyon, Marseille : ville entière et arrondissements présents en même temps ?
codes = pd.Series(delinq["CODGEO_2026"].unique())
print("Villes entières :", codes[codes.isin(["75056", "69123", "13055"])].tolist())
print("Arrondissements :", codes.str.match(r"^(751|6938|132)").sum())

# Test de rattachement sur la liste des communes
codes_delinq = delinq[["CODGEO_2026"]].drop_duplicates()
codes_perdus_delinq = tester_rattachement(codes_delinq, "CODGEO_2026", liaison_2026, "CODGEO_2026")
codes_perdus_delinq.head(50)

Codes distincts : 34920
Années : [np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
Indicateurs : 15
Villes entières : ['13055', '69123', '75056']
Arrondissements : 45
Lignes avant / après : 34920 / 34920
Lignes sans bassin   : 45


CODGEO_2026
13201    1
13202    1
13203    1
13204    1
13205    1
13206    1
13207    1
13208    1
13209    1
13210    1
13211    1
13212    1
13213    1
13214    1
13215    1
13216    1
69381    1
69382    1
69383    1
69384    1
69385    1
69386    1
69387    1
69388    1
69389    1
75101    1
75102    1
75103    1
75104    1
75105    1
75106    1
75107    1
75108    1
75109    1
75110    1
75111    1
75112    1
75113    1
75114    1
75115    1
75116    1
75117    1
75118    1
75119    1
75120    1
Name: count, dtype: int64